# Bronze para Silver — geração distribuída

Uma linha por UC com solar. A Bronze entra como texto. Esta Silver converte potência, injeção e data, corrige a escala acima de 75 kWp quando o rendimento bruto está no intervalo de erro em Wp (`<= 0,35` kWh/kWp/mês) e marca a conexão inconsistente.

A média injetada fica na linha para classificar a UC. Ela não é subtraída do consumo nem da fatura. A Gold é quem monta `possui_gd` a partir do mês de `data_conexao`.

As regras estão em `docs/silver/silver_geracao_distribuida.md`.


In [ ]:
# ruff: noqa: F821
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window

catalog = 'v-energy'
bronze_gd = f'`{catalog}`.bronze.tb_geracao_distribuida'
bronze_uc = f'`{catalog}`.bronze.tb_unidades_consumidoras'
silver_gd = f'`{catalog}`.silver.tb_geracao_distribuida'
quarentena_gd = f'`{catalog}`.silver.tb_geracao_distribuida_quarentena'

DATA_MIN = '2024-01-01'
PADRAO_DATA_ISO = r'^\d{4}-\d{2}-\d{2}$'
ESCALA_ACIMA_DE = 75
RAZAO_ESCALA_MAX = 0.35
AMOSTRA_DATA_INCONSISTENTE = [
    'UC100259',
    'UC102507',
    'UC106046',
    'UC110560',
    'UC119180',
]

MODALIDADE = {
    'GERACAO NA PROPRIA UC': 'PROPRIA_UC',
    'AUTOCONSUMO REMOTO': 'AUTOCONSUMO_REMOTO',
    'GERACAO COMPARTILHADA': 'COMPARTILHADA',
}

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.silver')
print(bronze_gd)
print(silver_gd)

In [ ]:
ACENTOS_ORIGEM = 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ'
ACENTOS_DESTINO = 'AAAAAEEEEIIIIOOOOOUUUUCN'


def limpar_espacos(col):
    return F.regexp_replace(F.trim(col), r'\s+', ' ')


def texto_padrao(col):
    limpo = limpar_espacos(col)
    sem_acento = F.translate(F.upper(limpo), ACENTOS_ORIGEM, ACENTOS_DESTINO)
    return F.initcap(sem_acento)


def converter_data_iso(col):
    texto = F.trim(col)
    convertida = F.try_to_date(texto, 'yyyy-MM-dd')
    return F.when(texto.rlike(PADRAO_DATA_ISO), convertida)


def calcular_regras_gd(potencia_num, energia_num, data_conexao, data_ligacao):
    razao_bruta = energia_num / potencia_num
    erro_escala = (potencia_num > F.lit(ESCALA_ACIMA_DE)) & (razao_bruta <= F.lit(RAZAO_ESCALA_MAX))
    potencia_corrigida = (
        F.when(erro_escala, potencia_num / F.lit(1000))
        .otherwise(potencia_num)
        .cast(DecimalType(10, 2))
    )
    rendimento = (energia_num / potencia_corrigida).cast(DecimalType(10, 2))
    fora_da_janela = (data_conexao < F.to_date(F.lit(DATA_MIN))) | (data_conexao > F.current_date())
    anterior_a_ligacao = data_ligacao.isNotNull() & (data_conexao < data_ligacao)
    data_inconsistente = fora_da_janela | anterior_a_ligacao
    return erro_escala, potencia_corrigida, rendimento, data_inconsistente


mapa_modalidade = F.create_map(
    [item for par in MODALIDADE.items() for item in (F.lit(par[0]), F.lit(par[1]))]
)

versao_bronze_gd = spark.sql(f'DESCRIBE HISTORY {bronze_gd} LIMIT 1').first()['version']
versao_bronze_uc = spark.sql(f'DESCRIBE HISTORY {bronze_uc} LIMIT 1').first()['version']
df_bronze_gd = spark.sql(f'SELECT * FROM {bronze_gd} VERSION AS OF {versao_bronze_gd}')
df_bronze_uc = spark.sql(f'SELECT * FROM {bronze_uc} VERSION AS OF {versao_bronze_uc}')

# A Bronze de UC repete o cabeçalho numa linha. Datas fora de yyyy-MM-dd viram nulo.
df_uc_raw = df_bronze_uc.filter(F.upper(F.trim(F.col('id_uc'))) != F.lit('ID_UC')).select(
    F.upper(F.trim(F.col('id_uc'))).alias('uc_id'),
    F.trim(F.col('data_ligacao')).alias('data_ligacao_raw'),
    converter_data_iso(F.col('data_ligacao')).alias('data_ligacao'),
    texto_padrao(F.col('classe')).alias('classe'),
    texto_padrao(F.col('municipio')).alias('municipio'),
    F.col('ingestion_datetime').alias('uc_ingestion_datetime'),
    F.col('source_file').alias('uc_source_file'),
)
hash_uc = F.sha2(
    F.concat_ws(
        '||',
        F.coalesce(F.col('uc_id'), F.lit('')),
        F.coalesce(F.col('data_ligacao_raw'), F.lit('')),
        F.coalesce(F.col('classe'), F.lit('')),
        F.coalesce(F.col('municipio'), F.lit('')),
    ),
    256,
)
janela_uc = Window.partitionBy('uc_id').orderBy(
    F.col('uc_ingestion_datetime').desc_nulls_last(),
    F.col('data_ligacao').desc_nulls_last(),
    F.col('uc_source_file').desc_nulls_last(),
    hash_uc.desc(),
)
df_uc = (
    df_uc_raw.withColumn('_rn', F.row_number().over(janela_uc))
    .filter(F.col('_rn') == 1)
    .drop('_rn', 'uc_ingestion_datetime', 'uc_source_file')
)
n_ligacao_invalida = df_uc.filter(
    F.col('data_ligacao_raw').isNotNull()
    & (F.col('data_ligacao_raw') != '')
    & F.col('data_ligacao').isNull()
).count()
if n_ligacao_invalida:
    print(f'[AVISO] UC com data_ligacao inválida: {n_ligacao_invalida}')
df_uc = df_uc.drop('data_ligacao_raw')

expr_potencia = "try_cast(regexp_replace(trim(potencia_kwp), ',', '.') as decimal(10,2))"
expr_energia = (
    "try_cast(regexp_replace(trim(energia_injetada_kwh_media_mensal), ',', '.') as decimal(10,2))"
)

df = df_bronze_gd.select(
    F.upper(F.trim(F.col('id_uc'))).alias('id_uc'),
    F.trim(F.col('potencia_kwp')).alias('potencia_kwp_raw'),
    F.expr(expr_potencia).alias('_potencia_num'),
    F.trim(F.col('energia_injetada_kwh_media_mensal')).alias('energia_raw'),
    F.expr(expr_energia).alias('_energia_num'),
    F.trim(F.col('data_conexao')).alias('data_conexao_raw'),
    converter_data_iso(F.col('data_conexao')).alias('data_conexao'),
    F.trim(F.col('modalidade')).alias('modalidade_origem'),
    F.col('source_file'),
    F.col('ingestion_datetime').alias('bronze_ingestion_datetime'),
).join(df_uc, F.col('id_uc') == F.col('uc_id'), 'left')

modalidade = mapa_modalidade[F.col('modalidade_origem')]
motivo = (
    F.when(F.col('id_uc').isNull() | (F.col('id_uc') == ''), F.lit('id_uc_nulo'))
    .when(
        F.col('_potencia_num').isNull() | (F.col('_potencia_num') <= 0),
        F.lit('potencia_invalida'),
    )
    .when(
        F.col('_energia_num').isNull() | (F.col('_energia_num') <= 0),
        F.lit('energia_injetada_invalida'),
    )
    .when(F.col('data_conexao').isNull(), F.lit('data_conexao_invalida'))
    .when(modalidade.isNull(), F.lit('modalidade_desconhecida'))
    .when(F.col('uc_id').isNull(), F.lit('uc_inexistente'))
)
df = df.withColumn('modalidade', modalidade).withColumn('motivo_rejeicao', motivo)

df_quarentena = df.filter(F.col('motivo_rejeicao').isNotNull())
df_ok = df.filter(F.col('motivo_rejeicao').isNull())

_, _, _, flag_data_dedup = calcular_regras_gd(
    F.col('_potencia_num'),
    F.col('_energia_num'),
    F.col('data_conexao'),
    F.col('data_ligacao'),
)
hash_gd = F.sha2(
    F.concat_ws(
        '||',
        F.col('id_uc'),
        F.col('potencia_kwp_raw'),
        F.col('energia_raw'),
        F.col('data_conexao_raw'),
        F.col('modalidade_origem'),
        F.coalesce(F.col('source_file'), F.lit('')),
    ),
    256,
)
janela_id = Window.partitionBy('id_uc').orderBy(
    flag_data_dedup.asc(),
    F.col('data_conexao').desc_nulls_last(),
    F.col('bronze_ingestion_datetime').desc_nulls_last(),
    F.col('source_file').desc_nulls_last(),
    hash_gd.desc(),
)
df_ok = df_ok.withColumn('_rn', F.row_number().over(janela_id)).withColumn(
    '_qtd_id', F.count('*').over(Window.partitionBy('id_uc'))
)
n_descartadas = df_ok.filter(F.col('_rn') > 1).count()
df_uma = df_ok.filter(F.col('_rn') == 1)

(
    erro_escala,
    potencia_corrigida,
    rendimento,
    data_inconsistente,
) = calcular_regras_gd(
    F.col('_potencia_num'),
    F.col('_energia_num'),
    F.col('data_conexao'),
    F.col('data_ligacao'),
)

df_silver = (
    df_uma.withColumn('potencia_kwp', potencia_corrigida)
    .withColumn('fl_correcao_escala', erro_escala)
    .withColumn(
        'energia_injetada_kwh_media_mensal',
        F.col('_energia_num').cast(DecimalType(10, 2)),
    )
    .withColumn('kwh_por_kwp_mes', rendimento)
    .withColumn('fl_data_inconsistente', data_inconsistente)
    .withColumn('fl_deduplicado', F.col('_qtd_id') > 1)
    .withColumn('silver_ingestion_datetime', F.current_timestamp())
    .select(
        'id_uc',
        'potencia_kwp_raw',
        'potencia_kwp',
        'fl_correcao_escala',
        'energia_injetada_kwh_media_mensal',
        'kwh_por_kwp_mes',
        'data_conexao',
        'fl_data_inconsistente',
        'modalidade',
        'modalidade_origem',
        'fl_deduplicado',
        'classe',
        'municipio',
        'source_file',
        'bronze_ingestion_datetime',
        'silver_ingestion_datetime',
    )
)


def validar_silver(df_alvo, df_uc_ref, rotulo):
    falhas = []

    def checar(nome, condicao):
        print(f'[{"OK" if condicao else "FALHA"}] {rotulo} — {nome}')
        if not condicao:
            falhas.append(nome)

    pot_orig = F.expr("try_cast(regexp_replace(trim(potencia_kwp_raw), ',', '.') as decimal(10,2))")
    com_uc = df_alvo.join(
        df_uc_ref.select('uc_id', 'data_ligacao'),
        F.col('id_uc') == F.col('uc_id'),
        'left',
    )
    (
        espera_escala,
        pot_esperada,
        rend_esperado,
        espera_data,
    ) = calcular_regras_gd(
        pot_orig,
        F.col('energia_injetada_kwh_media_mensal'),
        F.col('data_conexao'),
        F.col('data_ligacao'),
    )

    checar(
        'uma linha por id_uc',
        df_alvo.select('id_uc').distinct().count() == df_alvo.count(),
    )
    checar(
        'potência positiva',
        df_alvo.filter(F.col('potencia_kwp').isNull() | (F.col('potencia_kwp') <= 0)).count() == 0,
    )
    checar(
        'injeção positiva',
        df_alvo.filter(
            F.col('energia_injetada_kwh_media_mensal').isNull()
            | (F.col('energia_injetada_kwh_media_mensal') <= 0)
        ).count()
        == 0,
    )
    checar(
        'modalidade no de-para',
        df_alvo.filter(~F.col('modalidade').isin(list(MODALIDADE.values()))).count() == 0,
    )
    checar(
        'flag de escala segue a regra de potência e razão',
        df_alvo.filter(F.col('fl_correcao_escala') != espera_escala).count() == 0,
    )
    checar(
        'valor de potencia_kwp e kwh_por_kwp_mes bate com a transformação',
        df_alvo.filter(
            (F.col('potencia_kwp') != pot_esperada) | (F.col('kwh_por_kwp_mes') != rend_esperado)
        ).count()
        == 0,
    )
    checar(
        'linhas corrigidas ficam em [1, 75] kWp e rendimento >= 1',
        df_alvo.filter(
            F.col('fl_correcao_escala')
            & (
                (F.col('potencia_kwp') < 1)
                | (F.col('potencia_kwp') > ESCALA_ACIMA_DE)
                | (F.col('kwh_por_kwp_mes') < 1)
            )
        ).count()
        == 0,
    )
    checar(
        'flag de data segue a regra de janela e data_ligacao',
        com_uc.filter(F.col('fl_data_inconsistente') != espera_data).count() == 0,
    )

    if falhas:
        raise RuntimeError(f'[ERRO] validação ({rotulo}): ' + ', '.join(falhas))


n_bronze = df_bronze_gd.count()
n_silver = df_silver.count()
n_quarentena = df_quarentena.count()

if n_silver + n_quarentena + n_descartadas != n_bronze:
    raise RuntimeError(
        f'[ERRO] Bronze {n_bronze} não fecha com Silver {n_silver} + '
        f'quarentena {n_quarentena} + duplicatas {n_descartadas}.'
    )

validar_silver(df_silver, df_uc, 'antes de gravar')

if n_quarentena:
    print(f'[AVISO] quarentena: {n_quarentena} linha(s). A Silver das linhas válidas segue.')
    df_quarentena.groupBy('motivo_rejeicao').count().show(truncate=False)
else:
    print('[OK] quarentena: 0')
if n_descartadas:
    print(
        f'[AVISO] duplicatas descartadas: {n_descartadas}. fl_deduplicado marca a linha que ficou.'
    )

In [ ]:
df_quarentena_out = df_quarentena.select(
    'id_uc',
    'potencia_kwp_raw',
    'energia_raw',
    'data_conexao_raw',
    'modalidade_origem',
    'source_file',
    'bronze_ingestion_datetime',
    'motivo_rejeicao',
).withColumn('silver_ingestion_datetime', F.current_timestamp())


def versao_atual(tabela):
    if not spark.catalog.tableExists(tabela):
        return None
    return spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()['version']


def restaurar_tabela(tabela, versao_anterior):
    if not spark.catalog.tableExists(tabela):
        return
    if versao_anterior is None:
        spark.sql(f'DROP TABLE IF EXISTS {tabela}')
    else:
        spark.sql(f'RESTORE TABLE {tabela} TO VERSION AS OF {versao_anterior}')


versao_ant_silver = versao_atual(silver_gd)
versao_ant_quarentena = versao_atual(quarentena_gd)

try:
    (
        df_silver.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(silver_gd)
    )
    (
        df_quarentena_out.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(quarentena_gd)
    )
except Exception:
    restaurar_tabela(silver_gd, versao_ant_silver)
    restaurar_tabela(quarentena_gd, versao_ant_quarentena)
    raise

print(f'Silver gravada: {silver_gd} ({n_silver} linhas)')
print(f'Quarentena gravada: {quarentena_gd} ({n_quarentena} linhas)')

In [ ]:
gravada = spark.table(silver_gd)
gravada_quarentena = spark.table(quarentena_gd)
try:
    if gravada.count() != n_silver:
        raise RuntimeError(
            f'[ERRO] Silver gravada com {gravada.count()} linhas, esperado {n_silver}.'
        )
    if gravada_quarentena.count() != n_quarentena:
        raise RuntimeError(
            f'[ERRO] Quarentena gravada com {gravada_quarentena.count()} '
            f'linhas, esperado {n_quarentena}.'
        )
    validar_silver(gravada, df_uc, 'tabela gravada')
except Exception:
    restaurar_tabela(silver_gd, versao_ant_silver)
    restaurar_tabela(quarentena_gd, versao_ant_quarentena)
    raise

# Conferência informativa da amostra desta entrega. Se uma dessas UCs tiver
# a data corrigida na origem, a carga segue; aqui só avisa.
marcadas_amostra = {
    r['id_uc']
    for r in gravada.filter(
        F.col('id_uc').isin(AMOSTRA_DATA_INCONSISTENTE) & F.col('fl_data_inconsistente')
    )
    .select('id_uc')
    .collect()
}
if marcadas_amostra == set(AMOSTRA_DATA_INCONSISTENTE):
    print('[OK] amostra histórica: as 5 UCs conhecidas seguem marcadas.')
else:
    print(
        f'[AVISO] amostra histórica mudou: '
        f'marcadas={sorted(marcadas_amostra)}. '
        'A trava principal já conferiu a regra de datas.'
    )

print('Validação da tabela gravada fechou.')
gravada.groupBy('modalidade').count().orderBy('modalidade').show(truncate=False)